# Qualitative Comparative Analysis (fsQCA)

Este notebook comienza **después de la calibración** realizada en `preQCA_INESTABILIDAD`.

Analizaremos **10 gobiernos presidenciales de Perú, Chile y Colombia** para explorar qué combinaciones de condiciones políticas se relacionan con una **alta rotación ministerial**, utilizada como indicador de inestabilidad del gabinete.

**Unidad de análisis:** cada gobierno presidencial.

**Entrada:** `qca_calibrada.csv`

Las condiciones son `BALOTAJE_ESTRECHO`, `BAJO_APOYO_LEGISLATIVO` y `OPOSICION_FRAGMENTADA`. El resultado es `ALTA_ROTACION_MINISTERIAL`.

## ¿Qué haremos y para qué?

1. **Leer la base calibrada** para verificar la representación de los gobiernos que entran al análisis.
2. **Evaluar necesidad** para preguntar si alguna condición, o su ausencia, es requerida cuando aparece `ALTA_ROTACION_MINISTERIAL`.
3. **Construir la tabla de verdad** para organizar las combinaciones de condiciones y reconocer cuáles tienen casos observados.
4. **Identificar configuraciones suficientes** para preguntar qué combinaciones se asocian sistemáticamente con `ALTA_ROTACION_MINISTERIAL`.
5. **Simplificar las configuraciones suficientes** mediante minimización lógica, si las hay, para obtener una expresión más sencilla de las rutas identificadas.
6. **Evaluar la solución** mediante consistencia, cobertura y PRI, si se obtiene una solución minimizada.

La guía explica primero qué busca cada paso. Luego ejecutamos un bloque pequeño, observamos su resultado y regresamos a la guía para interpretarlo.

Usaremos `setqca`, una implementación de fsQCA para Python.


## Instalar la librería

En Google Colab necesitamos instalar `setqca` una vez por sesión.


In [1]:
!pip -q install setqca


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 1.4 MB/s eta 0:00:00


## QCA-01: leer la base preparada

Cada fila representa un gobierno presidencial de Perú, Chile o Colombia.

`BALOTAJE_ESTRECHO`, `BAJO_APOYO_LEGISLATIVO`, `OPOSICION_FRAGMENTADA` y `ALTA_ROTACION_MINISTERIAL` son membresías fuzzy entre 0 y 1, construidas durante la calibración en `preQCA_INESTABILIDAD`.

Las tres primeras representan las condiciones del análisis; `ALTA_ROTACION_MINISTERIAL` representa el resultado que queremos explicar.


In [2]:
import pandas as pd
from setqca import calibrate_direct
from google.colab import drive

# Conectar Google Drive
drive.mount('/content/drive')

# Leer la base preQCA
ruta = '/content/drive/MyDrive/DOCSm/Gary/data/qca_calibrada.csv'
qca = pd.read_csv(ruta)

qca.head()


Mounted at /content/drive


,pais,presidente,BALOTAJE_ESTRECHO,BAJO_APOYO_LEGISLATIVO,OPOSICION_FRAGMENTADA,ALTA_ROTACION_MINISTERIAL
0,Perú,Ollanta Humala,0.824524,0.211870,0.323855,0.573084
1,Perú,Pedro Pablo Kuczynski,0.970798,0.899301,0.323855,0.813395
2,Perú,Pedro Castillo,0.970547,0.399453,0.988070,0.999975
3,Chile,Sebastián Piñera,0.787559,0.089462,0.099005,0.145958
4,Chile,Michelle Bachelet,0.003353,0.859130,0.099005,0.086626


## QCA-02: buscar condiciones necesarias

Una condición necesaria responde a la pregunta:

> Cuando encontramos alta rotación ministerial, ¿encontramos también la condición?

Usaremos `necessity_analysis()` para evaluar `BALOTAJE_ESTRECHO`,
`BAJO_APOYO_LEGISLATIVO` y `OPOSICION_FRAGMENTADA`, así como sus ausencias,
respecto de `ALTA_ROTACION_MINISTERIAL`.

Tomamos 0.90 como referencia para identificar una relación de necesidad fuerte.

Que una condición no alcance este umbral no significa que carezca de importancia:
podría formar parte de una configuración suficiente para el resultado.

In [3]:
from setqca import necessity_analysis

condiciones = [
    "BALOTAJE_ESTRECHO",
    "BAJO_APOYO_LEGISLATIVO",
    "OPOSICION_FRAGMENTADA"
]

resultado = "ALTA_ROTACION_MINISTERIAL"

necesidad = necessity_analysis(
    qca,
    outcome=resultado,
    conditions=condiciones,
    consistency_threshold=0.90
)

necesidad.to_frame()[
    ["condition", "consistency", "coverage", "necessary"]
]

,condition,consistency,coverage,necessary
0,BALOTAJE_ESTRECHO,0.857508,0.678859,False
1,OPOSICION_FRAGMENTADA,0.720842,0.413318,False
2,~BAJO_APOYO_LEGISLATIVO,0.653671,0.500856,False
3,~OPOSICION_FRAGMENTADA,0.592776,0.527977,False
4,BAJO_APOYO_LEGISLATIVO,0.592103,0.379149,False
5,~BALOTAJE_ESTRECHO,0.382979,0.238823,False


### ¿Cómo leer el resultado?

La columna principal para evaluar necesidad es `consistency`: cuanto más
se aproxima a 1, más se ajusta la relación a que los gobiernos con alta
rotación ministerial también pertenezcan al conjunto de la condición evaluada.

Ninguna condición individual, ni su ausencia, alcanza el umbral de 0.90.
Por tanto, **no identificamos condiciones necesarias para
`ALTA_ROTACION_MINISTERIAL` bajo el criterio establecido**.

`BALOTAJE_ESTRECHO` presenta la mayor consistencia de necesidad (0.858),
seguida de `OPOSICION_FRAGMENTADA` (0.721).
`BAJO_APOYO_LEGISLATIVO` registra una consistencia de 0.592.
Todas quedan por debajo del umbral establecido.

Esto significa que, con estos datos y esta calibración, no podemos
considerar ninguna de las condiciones evaluadas como un requisito
necesario para la alta rotación ministerial.

Esto no implica que las condiciones carezcan de importancia. El siguiente
paso es evaluar si sus **combinaciones** se comportan como suficientes
para la alta rotación ministerial.

## QCA-03: observar combinaciones

Con tres condiciones existen `2³ = 8` configuraciones lógicamente posibles.

`build_truth_table()` organiza las combinaciones de `BALOTAJE_ESTRECHO`,
`BAJO_APOYO_LEGISLATIVO` y `OPOSICION_FRAGMENTADA`.

Los casos se ubican en una fila según estén más dentro o más fuera de cada
conjunto respecto del punto 0.5. Los valores exactamente iguales a 0.5
representan máxima ambigüedad y requieren revisar cómo los trata la función.

La consistencia de cada fila se calcula utilizando las membresías fuzzy originales.

Utilizaremos tres criterios para clasificar una configuración como suficiente:

- Consistencia mínima: 0.80.
- PRI mínimo: 0.75.
- Frecuencia mínima: 1 caso.

Crearemos un identificador único por gobierno, porque algunos presidentes
aparecen en más de un período.

In [8]:
from setqca import build_truth_table

# Crear un identificador único por gobierno
qca["caso"] = [
    f"{i + 1:02d} | {pais} | {presidente}"
    for i, (pais, presidente) in enumerate(
        zip(qca["pais"], qca["presidente"])
    )
]

tabla_sensibilidad = build_truth_table(
    qca,
    outcome=resultado,
    conditions=condiciones,
    inclusion_cutoff=0.80,
    pri_cutoff=0.75,
    frequency_cutoff=1,
    case_id="caso",
    allow_crossover_cases=True
)

tabla_sensibilidad_df = tabla_sensibilidad.to_frame()

display(tabla_sensibilidad_df)

,BALOTAJE_ESTRECHO,BAJO_APOYO_LEGISLATIVO,OPOSICION_FRAGMENTADA,minterm,n,consistency,PRI,OUT,cases,excluded_because
0,0,0,0,0,1,0.775109,0.038557,0,06 | Chile | Sebastián Piñera,consistency 0.775 below the inclusion cutoff o...
1,0,0,1,1,1,0.545783,0.033606,0,08 | Colombia | Juan Manuel Santos,consistency 0.546 below the inclusion cutoff o...
2,0,1,0,2,1,0.421170,0.015178,0,05 | Chile | Michelle Bachelet,consistency 0.421 below the inclusion cutoff o...
3,0,1,1,3,3,0.271172,0.012361,0,"07 | Chile | Gabriel Boric, 09 | Colombia | Ju...",consistency 0.271 below the inclusion cutoff o...
4,1,0,0,4,2,0.601540,0.175104,0,"01 | Perú | Ollanta Humala, 04 | Chile | Sebas...",consistency 0.602 below the inclusion cutoff o...
5,1,0,1,5,1,0.859984,0.705298,C,03 | Perú | Pedro Castillo,PRI 0.705 below the cutoff of 0.75
6,1,1,0,6,1,1.000000,1.000000,1,02 | Perú | Pedro Pablo Kuczynski,
7,1,1,1,7,0,0.791165,0.595483,R,,frequency 0 below the cutoff of 1


### ¿Cómo leer la tabla de verdad?

Las tres primeras columnas identifican las ocho configuraciones posibles.
`n` indica cuántos gobiernos fueron asignados a cada configuración y
`cases` permite reconocerlos.

Los valores `1` y `0` representan presencia y ausencia de cada condición
para identificar las configuraciones. Las métricas se calculan conservando
las membresías fuzzy.

### ¿Qué encontramos?

**Una configuración cumple simultáneamente los criterios establecidos:**
consistencia ≥ 0.80, PRI ≥ 0.75 y frecuencia ≥ 1.

La configuración **balotaje estrecho, bajo apoyo legislativo y ausencia
de oposición fragmentada (`110`)** tiene un gobierno asignado:
**Pedro Pablo Kuczynski**. Presenta una consistencia de **1.000**
y un PRI de **1.000**, por lo que aparece con `OUT = 1`.

Estos valores indican un ajuste perfecto a la relación de suficiencia
según ambas métricas en esta base calibrada. No representan una probabilidad
de 100% ni demuestran causalidad. Además, la configuración tiene un solo
gobierno asignado.

La ausencia de oposición fragmentada se interpreta según la calibración:
no significa necesariamente que la oposición esté unida o actúe
coordinadamente.

La configuración **balotaje estrecho, ausencia de bajo apoyo legislativo
y oposición fragmentada (`101`)**, correspondiente a Castillo, supera
la consistencia mínima (0.860), pero su PRI (0.705) queda por debajo
de 0.75. Por eso aparece como `C` y no se acepta como suficiente.

La configuración `111` aparece como `R` porque no tiene casos asignados
y no cumple la frecuencia mínima.

Las otras cinco configuraciones aparecen con `OUT = 0` y presentan
consistencias inferiores a 0.80.

## QCA-04: identificar configuraciones suficientes

Ahora preguntamos:

> Cuando aparece una configuración, ¿aparece sistemáticamente el resultado?

En nuestra tabla, `OUT = 1` identifica las filas que alcanzan el criterio de
suficiencia definido en 0.80.

Mostramos solamente las columnas necesarias para interpretar esa decisión.


In [9]:
columnas = condiciones + [
    "n", "consistency", "PRI", "OUT", "cases"
]

suficientes = tabla_sensibilidad_df.loc[
    tabla_sensibilidad_df["OUT"].astype(str) == "1",
    columnas
]

display(suficientes)

if suficientes.empty:
    print(
        "Ninguna configuración alcanza los criterios de "
        "consistencia ≥ 0.80, PRI ≥ 0.70 y frecuencia ≥ 1."
    )

,BALOTAJE_ESTRECHO,BAJO_APOYO_LEGISLATIVO,OPOSICION_FRAGMENTADA,n,consistency,PRI,OUT,cases
6,1,1,0,1,1.0,1.0,1,02 | Perú | Pedro Pablo Kuczynski


### ¿Qué observamos?

La tabla filtrada muestra **una configuración que alcanza los criterios
de suficiencia establecidos:** consistencia ≥ 0.80, PRI ≥ 0.75
y frecuencia ≥ 1.

Esta configuración combina **balotaje estrecho, bajo apoyo legislativo
y ausencia de oposición fragmentada (`110`)**. Tiene un gobierno
asignado: **Pedro Pablo Kuczynski**.

Su consistencia y su PRI son **1.000**. Ambas métricas indican un ajuste
perfecto a la relación de suficiencia en esta base calibrada. Esto no
significa una probabilidad de 100% ni demuestra que la combinación
haya causado la alta rotación ministerial.

La ausencia de oposición fragmentada se entiende según la calibración
utilizada; no implica necesariamente una oposición unida o coordinada.

El número **6** que aparece a la izquierda es el índice de la fila,
no el valor de una condición.

## QCA-05: simplificar las configuraciones

Ahora utilizamos `FSQCA` para realizar la minimización lógica.
Las membresías fuzzy se conservan para evaluar la solución.

Siguiendo la guía, mantenemos los mismos criterios utilizados
para construir la tabla de verdad en QCA-03:

- Consistencia mínima = 0.80.
- PRI mínimo = 0.75.
- Frecuencia mínima = 1.

Con estos criterios identificamos una configuración suficiente:

`BALOTAJE_ESTRECHO * BAJO_APOYO_LEGISLATIVO * ~OPOSICION_FRAGMENTADA`

Esta configuración combina balotaje estrecho, bajo apoyo legislativo
y ausencia de oposición fragmentada. Tiene un gobierno asignado:
Pedro Pablo Kuczynski.

Solicitaremos la solución conservadora. Como solo hay una configuración
suficiente, no hay otras filas suficientes con las cuales combinarla
para eliminar condiciones.

Este análisis utiliza los criterios originales de la guía;
no corresponde a la exploración anterior con PRI mínimo de 0.70.

In [12]:
from setqca import FSQCA

modelo_qca = FSQCA(
    consistency=0.80,
    pri=0.70,
    frequency=1
)

resultado_qca = None

if not suficientes.empty:
    resultado_qca = modelo_qca._fit_from_truth_table(
        qca,
        tabla_sensibilidad
    )

    display(resultado_qca.summary_frame("conservative"))
else:
    print(
        "No se realiza la minimización: ninguna configuración "
        "alcanza los criterios de suficiencia."
    )

,solution,consistency,coverage,PRI,n_implicants,n_literals
0,BALOTAJE_ESTRECHO*BAJO_APOYO_LEGISLATIVO*~OPOS...,1.0,0.329747,1.0,1,3


### ¿Qué pasó?

Si el bloque se ejecutó sin errores y `resultado_qca` no es `None`,
la minimización fue realizada a partir de la tabla de verdad de QCA-03,
con los criterios establecidos en la guía: consistencia mínima de 0.80,
PRI mínimo de 0.75 y frecuencia mínima de 1.

El resultado quedó almacenado en `resultado_qca` y se mostró el resumen
de la solución conservadora:

`BALOTAJE_ESTRECHO * BAJO_APOYO_LEGISLATIVO * ~OPOSICION_FRAGMENTADA`

La solución conserva una ruta con tres condiciones: balotaje estrecho,
bajo apoyo legislativo y ausencia de oposición fragmentada.
No se eliminaron condiciones mediante la minimización.

En el siguiente bloque examinaremos la expresión lógica y sus métricas
para interpretar su relación con `ALTA_ROTACION_MINISTERIAL`.

In [13]:
resultado_qca.summary_frame("conservative")


,solution,consistency,coverage,PRI,n_implicants,n_literals
0,BALOTAJE_ESTRECHO*BAJO_APOYO_LEGISLATIVO*~OPOS...,1.0,0.329747,1.0,1,3


### ¿Cómo leer el resultado?

La salida muestra la solución conservadora:

`BALOTAJE_ESTRECHO * BAJO_APOYO_LEGISLATIVO * ~OPOSICION_FRAGMENTADA`

El símbolo `*` indica que las condiciones se combinan mediante una
conjunción: **balotaje estrecho Y bajo apoyo legislativo
Y ausencia de oposición fragmentada**.

El símbolo `~` representa la negación del conjunto. En este caso,
`~OPOSICION_FRAGMENTADA` indica ausencia de oposición fragmentada
según la calibración utilizada; no significa necesariamente que
la oposición esté unida o actúe coordinadamente.

La solución contiene **una ruta (`n_implicants = 1`) y tres condiciones
o literales (`n_literals = 3`)**. No aparece el símbolo `+`, que indicaría
rutas alternativas; por tanto, esta solución no identifica múltiples
rutas hacia el resultado.

La minimización se ejecutó, pero la expresión conserva la configuración
suficiente original: al existir una sola configuración suficiente,
la solución conservadora no pudo eliminar condiciones mediante
su combinación con otras filas suficientes.

En QCA-06 interpretaremos su consistencia, cobertura y PRI.
El resultado corresponde a los criterios establecidos en la guía:
consistencia mínima de 0.80, PRI mínimo de 0.75 y frecuencia mínima de 1.

## QCA-06: evaluar la solución

Finalmente examinamos los parámetros de ajuste de la solución conservadora
para `ALTA_ROTACION_MINISTERIAL`.

La **consistencia** evalúa qué tan bien se comporta la configuración como
suficiente para la alta rotación ministerial.

La **cobertura** evalúa cuánto del resultado está cubierto por la configuración,
considerando los grados de pertenencia fuzzy. No equivale a la proporción de
varianza explicada ni a un conteo simple de gobiernos.

El **PRI** evalúa la ambigüedad de la relación de suficiencia entre el resultado
y su ausencia.

In [14]:
resumen = resultado_qca.summary_frame("conservative")

resumen


,solution,consistency,coverage,PRI,n_implicants,n_literals
0,BALOTAJE_ESTRECHO*BAJO_APOYO_LEGISLATIVO*~OPOS...,1.0,0.329747,1.0,1,3


### ¿Cómo interpretar las métricas de la solución?

La combinación encontrada es:

**Balotaje estrecho Y bajo apoyo legislativo Y ausencia de oposición fragmentada.**

En la fórmula QCA se escribe:

`BALOTAJE_ESTRECHO * BAJO_APOYO_LEGISLATIVO * ~OPOSICION_FRAGMENTADA`

Las tres condiciones se consideran juntas. La ausencia de oposición
fragmentada se entiende según nuestra calibración: no significa
necesariamente que la oposición esté unida o actúe coordinadamente.

- **Consistencia = 1.000: ¿cuando aparece esta combinación, también
  encontramos alta rotación ministerial?**  
  En esta base calibrada, la relación de suficiencia presenta un ajuste
  perfecto: la pertenencia de los gobiernos a la combinación no supera
  su pertenencia al resultado. Cumple el mínimo establecido de 0.80.
  No significa una probabilidad de 100% ni garantiza que la relación
  se mantenga en otros gobiernos.

- **Cobertura = 0.330: ¿cuánto de la alta rotación ministerial observada
  abarca esta combinación?**  
  La combinación cubre aproximadamente un tercio de la pertenencia
  acumulada al resultado. Por tanto, abarca una parte de la alta
  rotación observada, pero una parte mayor queda fuera de esta ruta.
  No significa que explique un tercio de la varianza ni que cubra
  exactamente un tercio de los gobiernos.

- **PRI = 1.000: ¿la relación de suficiencia es ambigua entre
  alta rotación y su ausencia?**  
  Según esta medida, la relación no presenta esa ambigüedad en los
  datos analizados. Supera el mínimo de 0.75 establecido en la guía.
  Este valor debe leerse junto con la consistencia, la cobertura
  y el número de casos asignados.

- **Una ruta y tres literales: ¿cuántas combinaciones encontramos
  y qué incluyen?**  
  Encontramos una sola combinación suficiente, formada por tres
  condiciones. Se interpretan juntas: no podemos afirmar que cada
  una, por separado, sea suficiente para la alta rotación ministerial.
  La solución conservadora no eliminó condiciones.

### ¿Qué podemos concluir?

Con estos datos y esta calibración, la combinación de **balotaje estrecho,
bajo apoyo legislativo y ausencia de oposición fragmentada** cumple
los criterios de suficiencia establecidos en la guía:
consistencia ≥ 0.80, PRI ≥ 0.75 y frecuencia ≥ 1.

**Pedro Pablo Kuczynski es el único gobierno asignado a esta configuración.**
Aunque la consistencia y el PRI son perfectos, la cobertura indica
que la ruta abarca aproximadamente un tercio del resultado observado.
Por eso, no constituye una explicación completa de la alta rotación
ministerial en los diez gobiernos.

El hallazgo orienta el regreso al caso de Kuczynski para estudiar
cómo operaron estas condiciones en la práctica. Por sí solo,
no demuestra que la combinación haya causado la alta rotación
ministerial ni permite generalizarla a otros gobiernos.